# 08 - Ensemble: Soft Voting

**Soft voting** = average the predicted class *probabilities* of several models, then take the argmax. It works when models are individually good AND make *different* mistakes, because averaging cancels uncorrelated errors.

This notebook:
1. **Measures error diversity** between models (which ones disagree -> worth combining).
2. Builds an **equal-weight soft vote** and picks the best member set by OOF macro-F1.
3. Adds **per-class threshold tuning** to lift the weak class 2 (macro-F1 weights every class equally).
4. Writes `outputs/ensemble_softvote_submission.csv`.

All scoring uses out-of-fold (OOF) probabilities on the SAME 5-fold split, so numbers are directly comparable to the single-model notebooks.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())   # make the local helper importable
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, classification_report
import ensemble_lib as eb        # shared: tuned base models + cached OOF/test probabilities

# Load each base model's probabilities. The FIRST run computes & caches them to
# outputs/preds/*.npy (slow, a few minutes); afterwards this is instant.
NAMES = ['svm', 'catboost', 'histgbm', 'logreg', 'knn']
oof  = {n: eb.get_proba(n)[0] for n in NAMES}   # leak-free train probabilities
tst  = {n: eb.get_proba(n)[1] for n in NAMES}   # test probabilities
print('\nIndividual OOF macro-F1 (baseline to beat):')
for n in NAMES:
    print(f'  {n:9s} {eb.macro_f1(oof[n]):.4f}')

[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities
[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities

Individual OOF macro-F1 (baseline to beat):
  svm       0.8324
  catboost  0.8216
  histgbm   0.8165
  logreg    0.7453
  knn       0.7641


## Step 1 - Error diversity
Soft voting only helps if members disagree. Below is the **disagreement matrix**: the fraction of training rows where two models predict different classes. Low values = redundant models (combining adds little); high values = diverse (combining helps).

Expectation from EDA: the two boosted trees (catboost, histgbm) should agree a lot with each other, while **svm** (a different model family) should be the outlier that disagrees most with the trees - making it the most valuable ensemble partner.

In [2]:
preds = {n: oof[n].argmax(1) for n in NAMES}
D = pd.DataFrame(index=NAMES, columns=NAMES, dtype=float)
for a in NAMES:
    for b in NAMES:
        D.loc[a, b] = (preds[a] != preds[b]).mean()
print('Pairwise disagreement (fraction of rows with different predictions):')
print(D.round(3))

Pairwise disagreement (fraction of rows with different predictions):
            svm  catboost  histgbm  logreg    knn
svm       0.000     0.091    0.116   0.180  0.168
catboost  0.091     0.000    0.093   0.197  0.161
histgbm   0.116     0.093    0.000   0.213  0.176
logreg    0.180     0.197    0.213   0.000  0.237
knn       0.168     0.161    0.176   0.237  0.000


**How to read it:** find the pair among the *strong* models (svm/catboost/histgbm) with the **highest** disagreement - that pair is your diversity backbone. catboost vs histgbm will be low (both boosted trees); svm vs either should be higher.

## Step 2 - Equal-weight soft vote
We try a few member sets and keep the one with the best OOF macro-F1. We specifically test whether adding histgbm (redundant with catboost) on top of svm+catboost actually helps, and whether the weak models (logreg/knn) help or hurt a plain average.

In [3]:
def vote(members, probs):
    return np.mean([probs[m] for m in members], axis=0)

combos = [
    ['svm', 'catboost'],
    ['svm', 'catboost', 'histgbm'],
    ['svm', 'catboost', 'histgbm', 'logreg'],
    ['svm', 'catboost', 'histgbm', 'logreg', 'knn'],
]
scores = {tuple(c): eb.macro_f1(vote(c, oof)) for c in combos}
for c, s in scores.items():
    print(f'{s:.4f}   {list(c)}')

MEMBERS = list(max(scores, key=scores.get))
print('\nBest member set:', MEMBERS, '-> OOF macro-F1 = %.4f' % scores[tuple(MEMBERS)])
print('Best single model was  : %.4f' % max(eb.macro_f1(oof[n]) for n in NAMES))

0.8317   ['svm', 'catboost']
0.8330   ['svm', 'catboost', 'histgbm']
0.8298   ['svm', 'catboost', 'histgbm', 'logreg']
0.8278   ['svm', 'catboost', 'histgbm', 'logreg', 'knn']

Best member set: ['svm', 'catboost', 'histgbm'] -> OOF macro-F1 = 0.8330
Best single model was  : 0.8324


## Step 3 - Per-class threshold tuning (orthogonal boost)
The metric is **macro**-F1, so the weak class 2 matters as much as any other. `tune_class_weights` multiplies each class's probability by a factor (found by coordinate ascent on the OOF score) before argmax - effectively lowering the bar to predict an under-predicted class. We tune on OOF and will apply the SAME factors to the test probabilities (never fit thresholds on test).

In [4]:
ens_oof = vote(MEMBERS, oof)
w, tuned = eb.tune_class_weights(ens_oof)
print('class weights:', np.round(w, 3))
print('OOF macro-F1  before tuning: %.4f' % eb.macro_f1(ens_oof))
print('OOF macro-F1  after  tuning: %.4f' % tuned)
print('\nPer-class report after tuning:')
print(classification_report(eb.y, (ens_oof * w).argmax(1), digits=3))

class weights: [0.85 1.1  1.2  0.85]
OOF macro-F1  before tuning: 0.8330
OOF macro-F1  after  tuning: 0.8357

Per-class report after tuning:
              precision    recall  f1-score   support

           0      0.878     0.846     0.862      2001
           1      0.843     0.862     0.853      2442
           2      0.772     0.812     0.791      2237
           3      0.857     0.819     0.837      2320

    accuracy                          0.835      9000
   macro avg      0.837     0.835     0.836      9000
weighted avg      0.836     0.835     0.835      9000



## Step 4 - Build the submission
Average the chosen members' TEST probabilities, apply the tuned class weights, argmax, and save.

In [5]:
os.makedirs('../outputs', exist_ok=True)
ens_test = vote(MEMBERS, tst)
final = (ens_test * w).argmax(1).astype(int)
sub = pd.DataFrame({'id': eb.test['id'], 'sleep_stage': final})
sub.to_csv('../outputs/ensemble_softvote_submission.csv', index=False)
print('wrote ../outputs/ensemble_softvote_submission.csv', sub.shape)
print('predicted class distribution:', dict(sub.sleep_stage.value_counts().sort_index()))
sub.head()

wrote ../outputs/ensemble_softvote_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1096), 1: np.int64(1316), 2: np.int64(1366), 3: np.int64(1222)}


,id,sleep_stage
0,9000,0
1,9001,3
2,9002,1
3,9003,2
4,9004,3


## Takeaways
- If the soft-vote OOF beats the best single model, the diversity was real (svm + a tree).
- Threshold tuning should nudge class-2 recall up at small cost elsewhere, raising macro-F1.
- Next: **09_weighted_soft_voting** (give stronger/decorrelated models more weight) and **10_stacking** (let a meta-model learn the best combination, including the weak members).